# Purpose/risk on gold1000-r4: inspect the run

Reads the finished run `gold1000-r4-2610130950-0` (current prompt, renderer-4
passages) and nothing else: no GPU, no model. The gold annotations were made on
gold1000-r1's text, so they are joined by `dsld_id` and shown for reference only;
their evidence spans do not occur verbatim in the r4 passages.

In [1]:
import json
import logging
import sys

import pandas as pd

from engine.repositories.dsld.labels import loader as label_sets
from engine.repositories.dsld.labels import paths as label_paths
from engine.repositories.dsld.runs import loader as runs
from engine.repositories.dsld.runs import paths as run_paths

RUN = run_paths.RUNS_ROOT / "gold1000-r4-2610130950-0"
GOLD_SET = "gold1000-r1"

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
pd.set_option("display.max_colwidth", 120)

## The run and its labels

In [2]:
manifest = runs.read_manifest(RUN)
print(manifest)

passages_sha, labels = label_sets.load_flattened(manifest["label_set"])
assert passages_sha == manifest["passages_sha"]

# No loader reads annotations yet (it comes with scoring), so the file is opened here.
with open(label_paths.set_dir(GOLD_SET) / label_paths.ANNOTATIONS_NAME) as f:
    gold = {int(r["dsld_id"]): r for r in map(json.loads, f)}

df = pd.DataFrame(runs.read_rows(RUN))
assert set(df.dsld_id) == set(labels) <= set(gold)
df[["dsld_id", "status", "attempts", "purpose", "risk", "latency_ms"]]

{'prompt': 'purpose_risk', 'label_set': 'gold1000-r4', 'passages_sha': '0801f354cfbd8472', 'selection': {'ids': None, 'limit': None, 'order': None, 'seed': None}, 'config_hash': 'dafc5e036ca02b9d'}


,dsld_id,status,attempts,purpose,risk,latency_ms
0,885,ok,1,[{'text': 'promotes general health and wellness'}],[{'text': 'if you are pregnant nursing or taking any medications consult your doctor before use'}],1865
1,1690,ok,1,[],[{'text': 'do not use if pregnant or nursing'}],894
2,709,ok,1,[{'text': 'supports memory retention'}],"[{'text': 'do not exceed recommended intake'}, {'text': 'consult your physicial prior to using this product if you a...",2874
3,889,ok,1,[{'text': 'support the health of the immune system'}],"[{'text': 'not intended for use by pregnant or nursing women'}, {'text': 'if you are taking any medications consult ...",4123
4,1736,ok,1,[],"[{'text': 'not intended for use by persons under the age of 18'}, {'text': 'if you are pregnant nursing taking any m...",2449
...,...,...,...,...,...,...
995,276703,ok,1,"[{'text': 'key nutrients critical for good looks health'}, {'text': 'delivering essential micro nutrients plus iron'...",[{'text': 'accidental overdose of iron containing products is a leading cause of fatal poisoning in children under 6...,3191
996,277101,ok,1,"[{'text': 'supports healthy circulation'}, {'text': 'helps with digestion'}, {'text': 'helps reduce feelings of anxi...","[{'text': 'possible skin sensitivity'}, {'text': 'if you are pregnant nursing or under a doctor s care consult your ...",2344
997,277412,ok,1,"[{'text': 'may help promote healthy hair skin and nails'}, {'text': 'it may also help support healthy hair skin and ...",[{'text': 'pregnant or nursing women individuals taking medication s or persons who have health conditions should co...,1748
998,277578,ok,1,[{'text': 'metabolic health'}],[{'text': 'if you are pregnant nursing taking any medications or have any medical condition consult your doctor befo...,1215


## Run stats

Status and attempts, items per label, and tokens and latency per label.

In [3]:
print(df.status.value_counts().to_string(), "\n")
print(df.attempts.value_counts().sort_index().rename("labels by attempts").to_string(), "\n")

df["n_purpose"] = df.purpose.str.len()
df["n_risk"] = df.risk.str.len()
print(df[["n_purpose", "n_risk"]].describe().round(2).to_string(), "\n")
print(f"no purpose: {(df.n_purpose == 0).sum()}   no risk: {(df.n_risk == 0).sum()}   "
      f"neither: {((df.n_purpose == 0) & (df.n_risk == 0)).sum()}   of {len(df)}\n")

# Slots run in parallel, so summed latency is not wall time.
print(df[["prompt_tokens", "completion_tokens", "latency_ms"]].describe().round(0).to_string())

status
ok    1000 

attempts
1    1000 

       n_purpose   n_risk
count    1000.00  1000.00
mean        2.35     1.69
std         2.81     1.71
min         0.00     0.00
25%         0.00     1.00
50%         1.00     1.00
75%         3.00     2.00
max        24.00    11.00 

no purpose: 256   no risk: 170   neither: 52   of 1000

       prompt_tokens  completion_tokens  latency_ms
count         1000.0             1000.0      1000.0
mean           773.0               56.0      2964.0
std            169.0               49.0      2192.0
min            561.0                2.0       407.0
25%            667.0               24.0      1581.0
50%            728.0               41.0      2397.0
75%            818.0               68.0      3672.0
max           2018.0              348.0     18802.0


## Are the items copied from the passage?

The prompt asks for claims in the label's own words. An item that is not a
substring of the passage was reworded or invented; those are listed for reading.

In [4]:
items = pd.DataFrame(
    [{"dsld_id": r.dsld_id, "kind": kind, "text": item["text"],
      "verbatim": item["text"] in labels[r.dsld_id].passage}
     for r in df.itertuples() for kind in ("purpose", "risk") for item in getattr(r, kind)]
)
print(items.groupby("kind").verbatim.agg(["sum", "count", "mean"]).round(3).to_string(), "\n")
items[~items.verbatim].head(30)

          sum  count   mean
kind                       
purpose  2304   2354  0.979
risk     1680   1690  0.994 



,dsld_id,kind,text,verbatim
33,2036,risk,consult your physician healthcare provider before beginning any exercise program,False
122,6616,purpose,support relaxation,False
123,6616,purpose,support overall well being,False
129,7688,risk,contains fish anchovy sardine mackerine and sardine ingredients,False
244,11807,purpose,enhances sexual energy in both men and women,False
364,18162,purpose,improve intensity,False
366,18162,purpose,amplify muscle pumps,False
651,30792,purpose,increases muscle mass,False
653,30792,purpose,increases endurance,False
855,36272,purpose,b 12 is important for healthy cardiovascular function,False


## Model beside gold: counts

Items per label from the model and from the gold, and how often each side has none
where the other has some. Only `ok` rows are counted.

In [5]:
ok = df[df.status == "ok"].copy()
ok["gold_purpose"] = ok.dsld_id.map(lambda i: len(gold[i]["purpose"]))
ok["gold_risk"] = ok.dsld_id.map(lambda i: len(gold[i]["risk"]))

print(ok[["n_purpose", "gold_purpose", "n_risk", "gold_risk"]].describe().round(2).to_string(), "\n")
for kind in ("purpose", "risk"):
    m, g = ok[f"n_{kind}"] > 0, ok[f"gold_{kind}"] > 0
    print(f"{kind:8s} both: {(m & g).sum():4d}   model only: {(m & ~g).sum():4d}   "
          f"gold only: {(~m & g).sum():4d}   neither: {(~m & ~g).sum():4d}   of {len(ok)}")

       n_purpose  gold_purpose   n_risk  gold_risk
count    1000.00       1000.00  1000.00     1000.0
mean        2.35          1.98     1.69        2.3
std         2.81          2.01     1.71        2.0
min         0.00          0.00     0.00        0.0
25%         0.00          0.00     1.00        1.0
50%         1.00          1.50     1.00        2.0
75%         3.00          3.00     2.00        3.0
max        24.00         15.00    11.00       14.0 

purpose  both:  698   model only:   46   gold only:   26   neither:  230   of 1000
risk     both:  808   model only:   22   gold only:  100   neither:   70   of 1000


## Sample: passage, model, gold

Each sampled label's r4 passage, the model's items, and the gold items for the same
label.

In [6]:
# A seeded handful of labels. Change SAMPLE_SEED to see others.
SAMPLE_SEED = 0

def show(rows):
    for row in rows.itertuples():
        label = labels[row.dsld_id]
        print(f"=== {row.dsld_id}  {label.name}  [{row.status}, {row.attempts} attempt(s)]")
        print(label.passage[:800] + (" ..." if len(label.passage) > 800 else ""))
        for kind in ("purpose", "risk"):
            print(f"--- {kind}: model")
            for item in getattr(row, kind):
                print(f"  {item['text']}")
            print(f"--- {kind}: gold")
            for item in gold[row.dsld_id][kind]:
                print(f"  {item['text']}")
        print()

show(df.sample(5, random_state=SAMPLE_SEED))

=== 276013  BioFit Max  [ok, 1 attempt(s)]
biofit max
suggested use as take capsules once daily for best results take minutes before a meal with an glass of water or as directed by your health care professional
probiotic 40 offers a balanced spectrum of live organisms probiotic bacteria are critical for healthy digestion help maintain the integrity of the intestinal lining support proper intestinal motility and participate in the detoxification process
this product is formulated with cfu per and to deliver a minimum potency of cfu per through the best by date
healthy gut flora support
contains cfu
caution as with any consult your healthcare practitioner before using this product especially if you are pregnant nursing anticipate surgery take medication on a regular basis or are otherwise under medical supervision
bi pass technology  ...
--- purpose: model
  probiotic bacteria are critical for healthy digestion
  help maintain the integrity of the intestinal lining
  support proper intes

## Where model and gold disagree most

Labels where the model found nothing but the gold has purpose items, and the
reverse.

In [7]:
show(ok[(ok.n_purpose == 0) & (ok.gold_purpose > 0)].head(3))
show(ok[(ok.n_purpose > 0) & (ok.gold_purpose == 0)].head(3))

=== 13711  Bee Pollen Caps 500 mg  [ok, 1 attempt(s)]
bee pollen caps
now r bee pollen delivers the natural nutrient profile found in genuine whole foods
consult physician if pregnant nursing taking medication or have a medical condition
capsule size
suggested usage as take to times daily preferably with food consider taking these other now r products spirulina lecithin and coq10
bee pollen is a fine powder that composes the male element of the flower honey bees harvesting the pollen instinctively gather only that pollen which is pure and healthy it is then mixed with flower nectar and bee saliva to form an all natural and highly nutritious food that has all twenty two of the elements composing the human body it supplies vitamins amino acids trace minerals and enzymes in a balance that provides optimum bioavailability
genuine whole food
nutri ...
--- purpose: model
--- purpose: gold
  nutrient rich superfood
  supplies vitamins, amino acids, trace minerals and enzymes in a balance that

## Most repeated purpose and risk items

In [8]:
# Repeated risk items are mostly generic label warnings.
for kind in ("purpose", "risk"):
    model_counts = pd.Series([item["text"] for items in df[kind] for item in items]).value_counts()
    gold_counts = pd.Series([item["text"] for r in gold.values() for item in r[kind]]).value_counts()
    print(f"--- {kind}: model {model_counts.sum()} items, {len(model_counts)} distinct   "
          f"gold {gold_counts.sum()} items, {len(gold_counts)} distinct")
    print(model_counts.head(10).to_string(), "\n")
    print(gold_counts.head(10).to_string(), "\n")

--- purpose: model 2354 items, 2137 distinct   gold 1979 items, 1845 distinct
immune support                      13
energy production                    9
antioxidant support                  6
supports joint health                6
immune system support                6
supports a healthy immune system     5
antioxidant                          5
bone support                         5
cardiovascular health                4
energy metabolism                    4 

immune support                    13
antioxidant support                7
immune system support              7
bone support                       5
antioxidant protection             4
energy production                  4
promotes cardiovascular health     4
supports bone health               4
heart health                       4
supports immune function           4 

--- risk: model 1690 items, 1070 distinct   gold 2302 items, 1285 distinct
if you are pregnant nursing taking any medications or have any medical condition co

## One failed label in full

In [9]:
failed = df[df.status != "ok"]
failed.iloc[0].to_dict() if len(failed) else "no failed labels"

'no failed labels'